In [2]:
import torch
import torch.nn as nn

In [3]:
torch.manual_seed(42)

# Parameters
batch_size = 2
seq_len = 5
input_size = 3
hidden_size = 4

In [4]:
# Input sequence
x = torch.randn(batch_size, seq_len, input_size)

In [5]:
# LSTM weights
W_ih = torch.randn(4 * hidden_size, input_size)
W_hh = torch.randn(4 * hidden_size, hidden_size)
b_ih = torch.randn(4 * hidden_size)
b_hh = torch.randn(4 * hidden_size)

In [6]:
# Manual LSTM
def manual_lstm(x, W_ih, W_hh, b_ih, b_hh):

    h = torch.zeros(x.size(0), hidden_size)
    c = torch.zeros(x.size(0), hidden_size)

    outputs = []

    for t in range(x.size(1)):

        xt = x[:, t, :]

        gates = (
            xt @ W_ih.T
            + h @ W_hh.T
            + b_ih
            + b_hh
        )

        i, f, g, o = gates.chunk(4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c = f * c + i * g
        h = o * torch.tanh(c)

        outputs.append(h.unsqueeze(1))

    return torch.cat(outputs, dim=1)


In [7]:
# Forward direction
forward_output = manual_lstm(
    x, W_ih, W_hh, b_ih, b_hh
)


In [8]:
# Backward direction
x_reverse = torch.flip(x, dims=[1])

backward_reverse = manual_lstm(
    x_reverse, W_ih, W_hh, b_ih, b_hh
)

backward_output = torch.flip(
    backward_reverse, dims=[1]
)


In [9]:
# Bidirectional output
bilstm_output = torch.cat(
    [forward_output, backward_output],
    dim=2
)


In [10]:
# Built-in PyTorch BiLSTM
builtin_lstm = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    batch_first=True,
    bidirectional=True
)

In [11]:
# Copy same weights
with torch.no_grad():

    builtin_lstm.weight_ih_l0.copy_(W_ih)
    builtin_lstm.weight_hh_l0.copy_(W_hh)
    builtin_lstm.bias_ih_l0.copy_(b_ih)
    builtin_lstm.bias_hh_l0.copy_(b_hh)

    builtin_lstm.weight_ih_l0_reverse.copy_(W_ih)
    builtin_lstm.weight_hh_l0_reverse.copy_(W_hh)
    builtin_lstm.bias_ih_l0_reverse.copy_(b_ih)
    builtin_lstm.bias_hh_l0_reverse.copy_(b_hh)

In [12]:

# Built-in output
builtin_output, _ = builtin_lstm(x)


In [13]:
# Validation
difference = torch.max(
    torch.abs(bilstm_output - builtin_output)
)

print("Input shape:", x.shape)
print("Manual BiLSTM shape:", bilstm_output.shape)
print("Built-in BiLSTM shape:", builtin_output.shape)
print("Maximum difference:", difference.item())
print("Outputs match:", torch.allclose(
    bilstm_output,
    builtin_output,
    atol=1e-6
))

Input shape: torch.Size([2, 5, 3])
Manual BiLSTM shape: torch.Size([2, 5, 8])
Built-in BiLSTM shape: torch.Size([2, 5, 8])
Maximum difference: 1.1920928955078125e-07
Outputs match: True
